In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CKPT = "/content/drive/MyDrive/genomic_best_model.pt"

ckpt = torch.load(CKPT, map_location=device, weights_only=False)

# Load config
CONFIG = ckpt["config"]
gene_ids = ckpt["gene_ids"]
scaler = ckpt["scaler"]
n_genes = ckpt["n_genes"]

In [10]:
import torch.nn as nn
import torch

class GenomicBERT(nn.Module):
    def __init__(self, n_genes, config):
        super().__init__()
        self.config = config

        self.cls_token = nn.Parameter(torch.randn(1, 1, config["hidden_size"])) # Class token for BERT-like models
        # From preprocess_single_tsv, input x is (batch_size, n_genes), each gene value is a single float.
        # So, gene_proj maps a single value (unsqueeze(-1)) to hidden_size.
        self.gene_proj = nn.Linear(1, config["hidden_size"])

        # +1 for the CLS token
        self.position_embeddings = nn.Embedding(n_genes + 1, config["hidden_size"])

        self.dropout = nn.Dropout(config["dropout"])

        # Transformer Encoder layers based on config
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config["hidden_size"],
            nhead=config["num_attention_heads"],
            dim_feedforward=config["intermediate_size"],
            dropout=config["dropout"],
            activation="gelu", # BERT often uses GELU activation
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config["num_hidden_layers"])

        # Pooler and Classifier for final prediction
        self.pooler = nn.Linear(config["hidden_size"], config["hidden_size"])
        self.pooler_activation = nn.Tanh()

        # Assuming binary classification (e.g., Tumor/Healthy)
        self.classifier = nn.Linear(config["hidden_size"], 2)

    def forward(self, x):
        # x shape: (batch_size, n_genes)
        batch_size = x.shape[0]

        # Project gene values
        # Unsqueeze to (batch_size, n_genes, 1) for the linear layer
        gene_embeddings = self.gene_proj(x.unsqueeze(-1)) # (batch_size, n_genes, hidden_size)

        # Expand CLS token to match batch size
        cls_token_expanded = self.cls_token.expand(batch_size, -1, -1) # (batch_size, 1, hidden_size)

        # Concatenate CLS token and gene embeddings
        embeddings = torch.cat((cls_token_expanded, gene_embeddings), dim=1) # (batch_size, n_genes + 1, hidden_size)

        # Add positional embeddings
        position_ids = torch.arange(embeddings.shape[1], device=x.device).unsqueeze(0)
        embeddings = embeddings + self.position_embeddings(position_ids)

        embeddings = self.dropout(embeddings)

        # Pass through transformer encoder
        encoded_output = self.encoder(embeddings) # (batch_size, n_genes + 1, hidden_size)

        # Get the output corresponding to the CLS token for classification
        cls_output = encoded_output[:, 0] # (batch_size, hidden_size)

        # Pooler
        pooled_output = self.pooler_activation(self.pooler(cls_output)) # (batch_size, hidden_size)

        # Classifier
        logits = self.classifier(pooled_output) # (batch_size, 2)
        return logits

model = GenomicBERT(n_genes, CONFIG).to(device)
model.load_state_dict(ckpt["model_state"], strict=False)
model.eval()

In [11]:
model = GenomicBERT(n_genes, CONFIG).to(device)
model.load_state_dict(ckpt["model_state"])   # NOW it will work
model.eval()

GenomicBERT(
  (gene_proj): Linear(in_features=1, out_features=256, bias=True)
  (pos_emb): Embedding(501, 256)
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.35, inplace=False)
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(1, 256, padding_idx=0)
      (position_embeddings): Embedding(502, 256)
      (token_type_embeddings): Embedding(2, 256)
      (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.35, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.35, inplace=False

In [12]:
import pandas as pd
import numpy as np

def preprocess_single_tsv(tsv_path, gene_ids, scaler, CONFIG):

    df = pd.read_csv(tsv_path, sep="\t", comment="#", low_memory=False)

    # Clean same as training
    df = df[~df["gene_id"].str.startswith("N_", na=False)]
    df = df[df["gene_id"] != "gene_id"]

    df = df.set_index("gene_id")

    # Select same expression column
    expr = pd.to_numeric(
        df[CONFIG["expression_col"]],
        errors="coerce"
    ).fillna(0.0)

    # Reindex to SAME genes used in training
    expr = expr.reindex(gene_ids, fill_value=0.0)

    # Convert to numpy
    x = expr.values.astype(np.float32)

    # Apply SAME transformations
    x = np.log1p(x)
    x = scaler.transform([x])[0]

    return torch.tensor(x, dtype=torch.float32).unsqueeze(0)

In [13]:
def predict_genomic(tsv_path):

    x = preprocess_single_tsv(tsv_path, gene_ids, scaler, CONFIG)
    x = x.to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)

    prob = probs[0][1].item()
    label = "Tumor" if prob > 0.5 else "Healthy"

    print("=================================")
    print(f"Prediction : {label}")
    print(f"Confidence : {prob:.4f}")
    print("=================================")

In [19]:
def predict_genomic(tsv_path):

    x = preprocess_single_tsv(tsv_path, gene_ids, scaler, CONFIG)
    x = x.to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)

    prob = probs[0][1].item()
    label = "Tumor" if prob > 0.5 else "Healthy"

    print("=================================")
    print(f"Prediction : {label}")
    print(f"Confidence : {prob:.4f}")
    print(f"Tumor Probability   : {prob:.4f}")
    print(f"Healthy Probability : {1 - prob:.4f}")
    print("=================================")

In [14]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/genomic_dataset.zip"
EXTRACT_PATH = "/content/genomic_data"

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("✅ Extracted to:", EXTRACT_PATH)

✅ Extracted to: /content/genomic_data


In [16]:
print(os.listdir("/content/genomic_data"))

['genomic data set']


In [17]:
import glob

tsv_files = glob.glob("/content/genomic_data/genomic data set/healthy/0d11c50a-8648-48ea-a107-e138a1d3e086/bf3ea4a0-bcd6-4e9d-acbb-3416f6ce53b7.rna_seq.augmented_star_gene_counts.tsv", recursive=True)

print("Total TSV files:", len(tsv_files))
print("Example file:", tsv_files[0])

Total TSV files: 1
Example file: /content/genomic_data/genomic data set/healthy/0d11c50a-8648-48ea-a107-e138a1d3e086/bf3ea4a0-bcd6-4e9d-acbb-3416f6ce53b7.rna_seq.augmented_star_gene_counts.tsv


In [20]:
predict_genomic(tsv_files[0])

Prediction : Healthy
Confidence : 0.2889
Tumor Probability   : 0.2889
Healthy Probability : 0.7111
